In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.pipeline import make_pipeline

In [2]:
df = pd.read_csv(
    "../Buoi3/du-lieu/training_set.csv",
    encoding="utf-8-sig"
)

print("Kích thước dữ liệu:", df.shape)
print("\n5 dòng đầu:")
print(df.head())

Kích thước dữ liệu: (1024, 3)

5 dòng đầu:
                                         van_ban_goc  \
0           lăo nhête sê kuô to pơ khâkhô neovuô khă   
1            gơi so tăosu nô nô muôlâ pơi côcê đăopư   
2  ngơibuô xi do khưkeo reobâ ho pư bâ ki côcê jơ...   
3                        nư sơ nhicâ câ nhơ điê khơi   
4                            tô khôsô nhicâ pê dư cê   

                                  ban_dich  nhan  
0    Cô gái dẫn trâu ra suối gần nhà rông.     1  
1                                      NaN     0  
2            Cây lúa dựng nhà ở buôn làng.     0  
3  Người thợ săn đang nấu cơm bên bờ suối.     0  
4               Người mẹ đi về trong rừng.     0  


In [3]:
X = df["van_ban_goc"].fillna("")
y = df["nhan"]

print("\nSố lượng mẫu:", len(X))
print("\nPhân bố nhãn:")
print(y.value_counts().sort_index())


Số lượng mẫu: 1024

Phân bố nhãn:
nhan
0    960
1     64
Name: count, dtype: int64


In [4]:
def f1_macro(y_true, y_pred):
    return f1_score(
        y_true,
        y_pred,
        average="macro",
        labels=[0, 1],
        zero_division=0
    )


# Scorer dành cho cross_val_score
cham = make_scorer(f1_macro)

In [5]:
Xh, Xk, yh, yk = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print("\nSố mẫu train:", len(Xh))
print("Số mẫu test :", len(Xk))


Số mẫu train: 819
Số mẫu test : 205


In [6]:
ket_qua = []

for ten, C, ng, mdf in (
    ("nho", 0.1, (2, 3), 5),
    ("vua", 4.0, (2, 4), 2),
    ("lon", 1000.0, (1, 7), 1)
):

    # --------------------------------------------------------
    # Tạo pipeline:
    # Text
    #   ↓
    # TF-IDF character n-gram
    #   ↓
    # Logistic Regression
    # --------------------------------------------------------

    m = make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=ng,
            min_df=mdf
        ),

        LogisticRegression(
            max_iter=4000,
            C=C,
            random_state=0
        )
    )

    # Train
    m.fit(Xh, yh)

    # Dự đoán train
    pred_hoc = m.predict(Xh)

    # Dự đoán test
    pred_kiem = m.predict(Xk)

    # F1 train
    a = f1_macro(yh, pred_hoc)

    # F1 test
    b = f1_macro(yk, pred_kiem)

    # Số lượng feature TF-IDF
    so_dt = len(
        m.named_steps["tfidfvectorizer"].vocabulary_
    )

    # Độ chênh lệch train - test
    chenh = a - b

    # Lưu kết quả
    ket_qua.append({
        "ten": ten,
        "C": C,
        "ngram_range": ng,
        "min_df": mdf,
        "so_feature": so_dt,
        "f1_train": a,
        "f1_test": b,
        "chenh_lech": chenh
    })

    print(
        "%-4s %6d dac trung  hoc %.4f  kiem %.4f  chenh %.4f"
        % (ten, so_dt, a, b, chenh)
    )

nho     909 dac trung  hoc 0.4839  kiem 0.4836  chenh 0.0003
vua    1647 dac trung  hoc 0.7718  kiem 0.5563  chenh 0.2155
lon    3613 dac trung  hoc 0.9726  kiem 0.5563  chenh 0.4163


In [7]:
best = max(
    ket_qua,
    key=lambda x: x["f1_test"]
)

print("\n================ KẾT QUẢ TỐT NHẤT ================")

print("Tên mô hình :", best["ten"])
print("C           :", best["C"])
print("ngram_range :", best["ngram_range"])
print("min_df      :", best["min_df"])
print("Số feature  :", best["so_feature"])
print("F1 train    :", round(best["f1_train"], 4))
print("F1 test     :", round(best["f1_test"], 4))
print("Chênh lệch  :", round(best["chenh_lech"], 4))


================ KẾT QUẢ TỐT NHẤT ================
Tên mô hình : vua
C           : 4.0
ngram_range : (2, 4)
min_df      : 2
Số feature  : 1647
F1 train    : 0.7718
F1 test     : 0.5563
Chênh lệch  : 0.2155


In [8]:
def mo_hinh():
    return make_pipeline(
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=best["ngram_range"],
            min_df=best["min_df"]
        ),

        LogisticRegression(
            max_iter=4000,
            C=best["C"],
            random_state=0
        )
    )

In [9]:
mot_lan = []
nhieu_lan = []

for s in (0, 1, 2, 42):

    print("\n---------- random_state =", s, "----------")

    # --------------------------------------------------------
    # 8.1. Chia train/test
    # --------------------------------------------------------

    Xh, Xk, yh, yk = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=s,
        stratify=y
    )

    # --------------------------------------------------------
    # 8.2. Train mô hình tốt nhất
    # --------------------------------------------------------

    m = mo_hinh()

    m.fit(Xh, yh)

    # --------------------------------------------------------
    # 8.3. F1 của một lần chia
    # --------------------------------------------------------

    pred = m.predict(Xk)

    diem_mot_lan = f1_macro(
        yk,
        pred
    )

    mot_lan.append(diem_mot_lan)

    print(
        "Một lần chia - F1:",
        round(diem_mot_lan, 4)
    )

    # --------------------------------------------------------
    # 8.4. 5-Fold Stratified Cross Validation
    # --------------------------------------------------------

    cvs = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=s
    )

    diem_cv = cross_val_score(
        mo_hinh(),
        X,
        y,
        cv=cvs,
        scoring=cham
    )

    diem_cv_mean = diem_cv.mean()

    nhieu_lan.append(diem_cv_mean)

    print(
        "5-Fold CV       :",
        round(diem_cv_mean, 4)
    )


---------- random_state = 0 ----------
Một lần chia - F1: 0.5563
5-Fold CV       : 0.5286

---------- random_state = 1 ----------
Một lần chia - F1: 0.4836
5-Fold CV       : 0.514

---------- random_state = 2 ----------
Một lần chia - F1: 0.4836
5-Fold CV       : 0.5129

---------- random_state = 42 ----------
Một lần chia - F1: 0.4836
5-Fold CV       : 0.514


In [10]:
print("\n================================================")
print("ĐÁNH GIÁ ĐỘ ỔN ĐỊNH")
print("================================================")

print(
    "F1 một lần chia:",
    [round(x, 4) for x in mot_lan]
)

print(
    "F1 5-Fold CV   :",
    [round(x, 4) for x in nhieu_lan]
)

print()

print(
    "Một lần chia: dai %.4f"
    % (max(mot_lan) - min(mot_lan))
)

print(
    "CV 5 phan    : dai %.4f"
    % (max(nhieu_lan) - min(nhieu_lan))
)


ĐÁNH GIÁ ĐỘ ỔN ĐỊNH
F1 một lần chia: [0.5563, 0.4836, 0.4836, 0.4836]
F1 5-Fold CV   : [np.float64(0.5286), np.float64(0.514), np.float64(0.5129), np.float64(0.514)]

Một lần chia: dai 0.0726
CV 5 phan    : dai 0.0156


In [11]:
model_final = mo_hinh()

model_final.fit(X, y)

print("\n================================================")
print("MÔ HÌNH CUỐI CÙNG")
print("================================================")

print("Đã train trên toàn bộ dữ liệu.")

print(
    "Số feature:",
    len(
        model_final
        .named_steps["tfidfvectorizer"]
        .vocabulary_
    )
)


MÔ HÌNH CUỐI CÙNG
Đã train trên toàn bộ dữ liệu.
Số feature: 1661
